# Gemma 4 E4B-it BC CPT - Dual Kaggle T4

This notebook prepares dual-GPU continued pretraining for `unsloth/gemma-4-E4B-it` on Business Central AL training text.

It uses Distributed Data Parallel through `torchrun --nproc_per_node=2`, following Unsloth's multi-GPU guidance. DDP creates one model copy per GPU, so each T4 must still fit the E4B-it 4-bit model. The defaults are intentionally conservative: `MAX_SEQ_LENGTH=4096`, `CHUNK_TOKENS=2048`, LoRA `r=32`, and embeddings/lm_head are off by default.

## Install

Run this in Kaggle with the dual T4 accelerator enabled. The training script will look for `business_central_al_training_text.txt` in attached Kaggle datasets.

In [ ]:
%%capture
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"

import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

!uv pip install -qqq     "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes     datasets accelerate unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm huggingface_hub

In [ ]:
!nvidia-smi

## Hugging Face Configuration

Set your Hugging Face account and token near the top before launching training. The training script reads these values from environment variables and can automatically push the LoRA adapter after training.

In [ ]:
import os

HF_ACCOUNT = "HF_ACCOUNT"  # Example: "anton"
HF_TOKEN = ""              # Paste token here, or leave blank to use Kaggle secret HF_TOKEN.
HF_REPO_NAME = "gemma-4-e4b-it-bc-cpt-lora"
PUSH_TO_HF = True

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = ""

if HF_ACCOUNT and HF_ACCOUNT != "HF_ACCOUNT":
    os.environ["HF_REPO_ID"] = f"{HF_ACCOUNT}/{HF_REPO_NAME}"
else:
    os.environ["HF_REPO_ID"] = ""

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["PUSH_TO_HF"] = "1" if PUSH_TO_HF and os.environ["HF_REPO_ID"] and HF_TOKEN else "0"

print("HF repo:", os.environ["HF_REPO_ID"] or "not configured")
print("HF push enabled:", os.environ["PUSH_TO_HF"] == "1")

## Write DDP Training Script

The script contains model loading, dataset chunking, baseline prompts before training, DDP SFT training, saving, optional HF push, and post-training prompt comparison.

In [ ]:
%%writefile train_e4b_it_dual_t4.py
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

from pathlib import Path
import torch
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastModel

MODEL_NAME = "unsloth/gemma-4-E4B-it"
OUTPUT_DIR = "gemma-4-e4b-it-bc-cpt-lora"
MAX_SEQ_LENGTH = 4096
CHUNK_TOKENS = 2048
OVERLAP_TOKENS = 128
MIN_CHUNK_TOKENS = 256
NUM_TRAIN_EPOCHS = 0.5
EVAL_STEPS = 50
SAVE_STEPS = 350
SEED = 3407

local_rank = int(os.environ.get("LOCAL_RANK", "0"))
world_size = int(os.environ.get("WORLD_SIZE", "1"))
is_main_process = local_rank == 0
if torch.cuda.is_available():
    torch.cuda.set_device(local_rank)


def log(message):
    if is_main_process:
        print(message, flush=True)


def find_training_text():
    explicit_path = os.environ.get("KAGGLE_TRAINING_TEXT_PATH")
    if explicit_path:
        path = Path(explicit_path)
        if path.exists():
            return path
        raise FileNotFoundError(f"KAGGLE_TRAINING_TEXT_PATH does not exist: {path}")

    candidates = sorted(Path("/kaggle/input").glob("**/business_central_al_training_text.txt"))
    if not candidates:
        candidates = sorted(Path("/kaggle/input").glob("**/corpus.txt"))
    if not candidates:
        raise FileNotFoundError(
            "Could not find business_central_al_training_text.txt under /kaggle/input. "
            "Attach the Kaggle Dataset that contains the generated training text."
        )
    return candidates[0]


def get_text_tokenizer(tokenizer):
    return getattr(tokenizer, "tokenizer", tokenizer)


def unused_token_ids(text_tokenizer, limit=256):
    ids = []
    for index in range(limit):
        token = f"<unused{index}>"
        token_id = text_tokenizer.convert_tokens_to_ids(token)
        if token_id is not None and token_id != text_tokenizer.unk_token_id:
            ids.append([token_id])
    return ids


def text_completion(model, tokenizer, prompt, max_new_tokens=160, temperature=0.2):
    text_tokenizer = get_text_tokenizer(tokenizer)
    device = f"cuda:{local_rank}" if torch.cuda.is_available() else "cpu"
    inputs = text_tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(device)
    do_sample = temperature is not None and temperature > 0
    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        do_sample=do_sample,
        repetition_penalty=1.05,
        eos_token_id=text_tokenizer.eos_token_id,
        pad_token_id=text_tokenizer.pad_token_id or text_tokenizer.eos_token_id,
        bad_words_ids=unused_token_ids(text_tokenizer),
    )
    if do_sample:
        generation_kwargs.update(temperature=temperature, top_p=0.9, top_k=40)

    with torch.inference_mode():
        output_ids = model.generate(**generation_kwargs)
    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return text_tokenizer.decode(generated_ids, skip_special_tokens=True)


def make_dataset(tokenizer):
    training_text_path = find_training_text()
    log(f"Training text path: {training_text_path}")

    raw_text = training_text_path.read_text(encoding="utf-8")
    text_tokenizer = get_text_tokenizer(tokenizer)
    token_ids = text_tokenizer.encode(raw_text, add_special_tokens=False)
    step = CHUNK_TOKENS - OVERLAP_TOKENS

    chunks = []
    for start in range(0, len(token_ids), step):
        chunk_ids = token_ids[start:start + CHUNK_TOKENS]
        if len(chunk_ids) < MIN_CHUNK_TOKENS and chunks:
            break
        chunks.append(text_tokenizer.decode(chunk_ids) + tokenizer.eos_token)

    dataset = Dataset.from_dict({"text": chunks})
    split_dataset = dataset.train_test_split(test_size=0.02, seed=SEED, shuffle=True)

    log(f"Training text characters: {len(raw_text):,}")
    log(f"Training text tokens: {len(token_ids):,}")
    log(f"Chunk tokens: {CHUNK_TOKENS:,}")
    log(f"Training examples: {len(split_dataset['train']):,}")
    log(f"Validation examples: {len(split_dataset['test']):,}")
    return split_dataset["train"], split_dataset["test"]


log(f"DDP world size: {world_size}; local rank: {local_rank}")
log(f"Loading {MODEL_NAME}")
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    dtype=None,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
    token=os.environ.get("HF_TOKEN") or None,
    device_map={"": local_rank},
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    # E4B-it on dual T4 is tight. Keep embeddings/lm_head off by default;
    # enable them only if this fits comfortably and you want stronger CPT.
    finetune_embedding_modules=False,
    finetune_lm_head=False,
    r=32,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
    use_rslora=True,
)

train_dataset, eval_dataset = make_dataset(tokenizer)

baseline_prompts = [
    '''tableextension 50100 CustomerExt extends Customer
{
    fields
    {
        field(50100; "Credit Risk Score"; Integer)
        {
            Caption = 'Credit Risk Score';''',
    '''codeunit 50100 SalesOrderValidation
{
    procedure ValidateCustomerCreditLimit(SalesHeader: Record "Sales Header")
    var
        Customer: Record Customer;
    begin''',
    '''pageextension 50100 CustomerCardExt extends "Customer Card"
{
    layout
    {
        addlast(General)
        {
            field("Credit Risk Score"; Rec."Credit Risk Score")''',
]

if is_main_process:
    FastModel.for_inference(model)
    print("BASELINE COMPLETIONS BEFORE TRAINING")
    print("=" * 50)
    for prompt in baseline_prompts:
        print(f"\nPrompt:\n{prompt}")
        print("-" * 50)
        print(text_completion(model, tokenizer, prompt, max_new_tokens=160, temperature=0.2))
    if hasattr(FastModel, "for_training"):
        FastModel.for_training(model)
    else:
        model.train()

if torch.distributed.is_available() and torch.distributed.is_initialized():
    torch.distributed.barrier()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=CHUNK_TOKENS + 1,
        packing=False,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_ratio=0.03,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=1e-5,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=SEED,
        report_to="none",
        ddp_find_unused_parameters=False,
        output_dir=OUTPUT_DIR,
    ),
)

trainer_stats = trainer.train()
log(trainer_stats)

if torch.distributed.is_available() and torch.distributed.is_initialized():
    torch.distributed.barrier()

if is_main_process:
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Saved LoRA adapter to {OUTPUT_DIR}")

    FastModel.for_inference(model)
    print("POST-TRAINING COMPLETIONS")
    print("=" * 50)
    for prompt in baseline_prompts:
        print(f"\nPrompt:\n{prompt}")
        print("-" * 50)
        print(text_completion(model, tokenizer, prompt, max_new_tokens=220, temperature=0.2))

    if os.environ.get("PUSH_TO_HF", "0") == "1":
        hf_repo_id = os.environ["HF_REPO_ID"]
        hf_token = os.environ.get("HF_TOKEN")
        model.push_to_hub(hf_repo_id, token=hf_token)
        tokenizer.push_to_hub(hf_repo_id, token=hf_token)
        print(f"Pushed LoRA adapter to https://huggingface.co/{hf_repo_id}")


## Run Dual-GPU Training

This launches two Python processes, one per T4. If E4B-it OOMs, reduce `CHUNK_TOKENS` to `1024`, reduce LoRA `r` to `16`, or fall back to E2B-it.

In [ ]:
!torchrun --standalone --nproc_per_node=2 train_e4b_it_dual_t4.py

## Output

The LoRA adapter is saved under `/kaggle/working/gemma-4-e4b-it-bc-cpt-lora` unless you enable automatic Hugging Face push.

For GGUF export, use a separate notebook/session after the LoRA adapter is safely saved or pushed. Exporting GGUF can take enough time that it is better treated as a separate step.